# Mobilitätsauslastungsprognose: Wetterdaten

## 1. Installation wichtiger Pakete

In [1]:
# Installiere die benötigten Pakete

# pip install openmeteo-requests requests-cache retry-requests pandas pyspark numpy pandas

## 2. Bibliotheken importieren

In [2]:
# Importiere Bibliotheken für Wetter-API-Abfragen und Caching
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Importiere Pandas für Datenmanipulation
import pandas as pd

# Importiere PySpark-Bibliotheken für Datenverarbeitung
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when

## 3. Spark Session erstellen

In [ ]:
# Falls noch keine Spark-Session existiert, wird eine neue erstellt
spark = SparkSession.builder.appName("Import Weather Data").getOrCreate()

## 4. Einlesen der historischen Wetterdaten

In [5]:
# Setup des Open-Meteo API Clients mit Cache und Retry-Mechanismus
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# API-Anfrage an das Open-Meteo Archive
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.9624,
    "longitude": 7.6257,
    "start_date": "2018-01-01",
    "end_date": "2025-04-10",
    "hourly": ["temperature_2m", "rain", "snowfall", "wind_speed_10m", "is_day"],
    "timezone": "Europe/Berlin"
}
responses = openmeteo.weather_api(url, params=params)

# Verarbeite die erste Antwort der API
response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()} {response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Verarbeite die stündlichen Wetterdaten
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(3).ValuesAsNumpy()
hourly_is_day = hourly.Variables(4).ValuesAsNumpy()

# Erstelle ein Dictionary mit stündlichen Daten basierend auf einem Datumsbereich
hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )
}
hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["is_day"] = hourly_is_day

# Erstelle einen Pandas DataFrame mit den stündlichen Wetterdaten
hourly_dataframe = pd.DataFrame(data=hourly_data)
print(hourly_dataframe)

## 5. Hinzufügen weiterer Wetterkennzahlen

In [6]:
# ## Konvertiere den Pandas DataFrame in einen Spark DataFrame
hourly_spark_df = spark.createDataFrame(hourly_dataframe)

# ## Konvertiere die Spalte "date" in einen Timestamp (falls sie noch nicht als Timestamp vorliegt)
hourly_spark_df = hourly_spark_df.withColumn("date", F.to_timestamp("date"))

# ## Schritt 1: Extrahiere "date_only" (Datum ohne Zeitanteil)
hourly_spark_df = hourly_spark_df.withColumn("date_only", F.to_date(F.col("date")))

# ## Schritt 2: Extrahiere die Stunde aus der "date"-Spalte
hourly_spark_df = hourly_spark_df.withColumn("hour", F.hour(F.col("date")))

# ## Füge verzögerte (lagged) Wetterinformationen für den Niederschlag (rain) hinzu
window_spec = Window.partitionBy("date_only").orderBy("date_only", "hour")
hourly_spark_df = hourly_spark_df.withColumn("rain_lag_1h", F.lag("rain", 1).over(window_spec))
hourly_spark_df = hourly_spark_df.withColumn("rain_lag_3h", F.lag("rain", 3).over(window_spec))
hourly_spark_df = hourly_spark_df.withColumn("rain_lag_6h", F.lag("rain", 6).over(window_spec))
hourly_spark_df = hourly_spark_df.withColumn("rain_lag_12h", F.lag("rain", 12).over(window_spec))

# ## Benenne Spalten um, um Konflikte zu vermeiden
result_dataframe = hourly_spark_df.withColumnRenamed("date_only", "date1")

# ## Definiere die neue Spaltenreihenfolge und wähle die gewünschten Spalten aus
new_column_order = [
    "date1", "hour", "temperature_2m", "rain", "rain_lag_1h", 
    "rain_lag_3h", "rain_lag_6h", "rain_lag_12h", "snowfall", 
    "wind_speed_10m", "is_day"
]
result_dataframe = result_dataframe.select(new_column_order)

# ## Benenne "date1" wieder in "date" um
result_dataframe = result_dataframe.withColumnRenamed("date1", "date")

# ## Ersetze Nullwerte in den Lag-Spalten durch den Wert aus der Spalte "rain"
result_dataframe = result_dataframe.withColumn("rain_lag_1h", F.coalesce(F.col("rain_lag_1h"), F.col("rain"))) \
                                   .withColumn("rain_lag_3h", F.coalesce(F.col("rain_lag_3h"), F.col("rain"))) \
                                   .withColumn("rain_lag_6h", F.coalesce(F.col("rain_lag_6h"), F.col("rain"))) \
                                   .withColumn("rain_lag_12h", F.coalesce(F.col("rain_lag_12h"), F.col("rain")))

# Zeige das finale DataFrame an
result_dataframe.show()

In [7]:
# 1. Create a categorical column "rain_cat" with 4 categories
result_dataframe = (
    result_dataframe 
    .withColumn(
        "rain_bins",
        when(col("rain") == 0, "No Rain")
        .when((col("rain") > 0) & (col("rain") < 2), "Low Rain")
        .when((col("rain") >= 2) & (col("rain") < 5), "Middle Rain")
        .otherwise("Heavy Rain")
    )
)

## 6. Agrregation der Daten zu einem Data Frame

In [8]:
result_dataframe = (
    result_dataframe
    # Ersetze Nullwerte in "rain_lag_1h" durch den Wert aus "rain"
    .withColumn("rain_lag_1h", when(col("rain_lag_1h").isNull(), col("rain")).otherwise(col("rain_lag_1h")))
    # Falls "rain_lag_3h" Null ist, ersetze ihn durch den Wert von "rain_lag_1h"
    .withColumn("rain_lag_3h", when(col("rain_lag_3h").isNull(), col("rain_lag_1h")).otherwise(col("rain_lag_3h")))
    # Falls "rain_lag_6h" Null ist, ersetze ihn durch den Wert von "rain_lag_3h"
    .withColumn("rain_lag_6h", when(col("rain_lag_6h").isNull(), col("rain_lag_3h")).otherwise(col("rain_lag_6h")))
    # Falls "rain_lag_12h" Null ist, ersetze ihn durch den Wert von "rain_lag_6h"
    .withColumn("rain_lag_12h", when(col("rain_lag_12h").isNull(), col("rain_lag_6h")).otherwise(col("rain_lag_12h")))
)

# Definiere sinnvolle Gewichtungen für die Verzögerungswerte
weights = [0.4, 0.3, 0.2, 0.1]  # Diese sollten entweder insgesamt 1 ergeben oder die relative Wichtigkeit widerspiegeln

# Füge eine Spalte für den gewichteten Durchschnitt des Regens hinzu und runde auf zwei Dezimalstellen
result_dataframe = result_dataframe.withColumn(
    "weighted_avg_rain",
    F.round(
        F.coalesce(F.col("rain_lag_1h") * weights[0], F.lit(0)) +
        F.coalesce(F.col("rain_lag_3h") * weights[1], F.lit(0)) +
        F.coalesce(F.col("rain_lag_6h") * weights[2], F.lit(0)) +
        F.coalesce(F.col("rain_lag_12h") * weights[3], F.lit(0)),
        2  # Runden auf zwei Dezimalstellen
    )
)

# Entferne die Hilfsspalten für die Verzögerungswerte, da diese nicht mehr benötigt werden
result_dataframe = result_dataframe.drop("rain_lag_1h", "rain_lag_3h", "rain_lag_6h", "rain_lag_12h")

# Ordne die finalen Spalten im DataFrame neu an und wähle diese aus
result_dataframe = result_dataframe.select(
    "date", "hour", "temperature_2m", "rain", "rain_bins", "weighted_avg_rain", "snowfall", "wind_speed_10m", "is_day"
)

# Zeige das finale DataFrame an
result_dataframe.show(truncate=False)

##  7. Export der Wetterdaten

In [9]:
# Definiere den Speicherpfad für die bereinigten Daten im Parquet-Format
save_full_path = 'data/weather_data'

# Speichere die bereinigten Daten als Parquet-Datei (Überschreiben des bestehenden Inhalts)
result_dataframe.write.mode("overwrite").parquet(save_full_path)

# Bestätige den erfolgreichen Speichervorgang
print(f"Daten erfolgreich gespeichert unter: {save_full_path}")